### 迪杰斯特拉算法

In [4]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra


# 定义一个有向图的邻接矩阵(0 表示没有边)
graph = [
    [0 , 1 , 4 , 0] ,
    [0 , 0 , 2 , 6] ,
    [0 , 0 , 0 , 3] ,
    [0 , 0 , 0 , 0] 
]

# 转换为 CSR 矩阵
graph_csr = csr_matrix(graph)

# indices = 0 表示计算从节点 0 出发到其他节点的最短路径

distances , predecessors = dijkstra(csgraph = graph_csr ,
                                    directed = True ,
                                    indices = 0 , 
                                    return_predecessors = True
                                    )
print("从节点 0 出发到其他节点的最短路径长度: " , distances)
print("从节点 0 出发到其他节点的最短路径前驱节点: " , predecessors)


从节点 0 出发到其他节点的最短路径长度:  [0. 1. 3. 6.]
从节点 0 出发到其他节点的最短路径前驱节点:  [-9999     0     1     2]


##### 结果解析
- distances: 从起点到每个节点的最短距离
* predecessors: 每个节点的前驱节点，用于重建路径
  * 值为 -9999 表示没有前驱节点（起点或不可达节点
  * 如果 predecessors[3] = 2，表示去往 3 的最短路径上，3 的前一个节点是 2

##### 常用参数说明
- directed: True 表示有向图，False 表示无向图
- indices: 指定起点 . 如果不指定，则会计算所有节点对之间的最短路径
- limit : 设置搜索的最大距离

### 弗洛伊德算法
- 迪杰斯特拉算法常用 0 表示不可达节点，而弗洛伊德算法常用 float('inf') 表示不可达节点

In [5]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import floyd_warshall

# 1. 定义有向图的邻接矩阵
# 注意：在弗洛伊德算法中，0 仅表示自环（点到自身）
# 必须用 np.inf 表示节点之间没有直接路径
INF = np.inf
graph = [
    [0,   1,   4,   INF], # 节点 0 到其他点的初始距离
    [INF, 0,   2,   6],   # 节点 1 到其他点的初始距离
    [INF, INF, 0,   3],   # 节点 2 到其他点的初始距离
    [INF, INF, INF, 0]    # 节点 3 到其他点的初始距离
]

# 2. 转换为 CSR 稀疏矩阵（可选，但在 SciPy 中处理图时这是推荐做法）
graph_csr = csr_matrix(graph)

# 3. 执行 Floyd-Warshall 算法
# directed=True:  考虑边的方向
# return_predecessors=True: 返回前驱矩阵，用于回溯具体的路径
distances, predecessors = floyd_warshall(
    csgraph=graph_csr, 
    directed=True, 
    return_predecessors=True
)

# --- 结果输出 ---

# 输出是一个 N x N 的矩阵，distances[i, j] 表示从 i 到 j 的最短距离
print("所有节点对之间的最短路径长度矩阵：\n", distances)

# predecessors[i, j] 表示从 i 到 j 的路径中，到达 j 之前的那个节点索引
print("\n所有节点对的最短路径前驱矩阵：\n", predecessors)

# 示例：查看从节点 0 到节点 3 的距离
print(f"\n从节点 0 到节点 3 的最短距离为: {distances[0, 3]}")

所有节点对之间的最短路径长度矩阵：
 [[ 0.  1.  3.  6.]
 [inf  0.  2.  5.]
 [inf inf  0.  3.]
 [inf inf inf  0.]]

所有节点对的最短路径前驱矩阵：
 [[-9999     0     1     2]
 [-9999 -9999     1     2]
 [-9999 -9999 -9999     2]
 [-9999 -9999 -9999 -9999]]

从节点 0 到节点 3 的最短距离为: 6.0


### 弗洛伊德与迪杰斯特算法的区别
- 迪杰斯特拉 , 解决单源最短路径问题 , 贪心算法
- 弗洛伊德算法 , 解决全源最短路径 , 动态规划算法

### 贝尔曼-福特算法
- 求解单源最短路径问题 , 包括带负权边的图
* 基本思想:
  * 如果图中有 V 个顶点 , 那么对图中所有的边进行 V - 1 轮检查
  * 每一轮检查都会问 : 从 A 到 B 如果经过这条边 , 距离会不会变短? 如果变短了 , 就更新它

##### 手写 贝尔曼-福特 算法

In [6]:
import numpy as np

def bellman_ford_custom(graph , start_node):
    n = len(graph)

    # 1 . 初始化距离 , 起点为 0 , 其他为无穷大
    distances = np.full(n , np.inf)
    distances[start_node] = 0

    # 2 . 核心 : 进行 n- 1 轮松弛
    # 每一轮都尝试更新图中的所有边
    for i in range(n - 1):
        for u in range(n):
            for v in range(n):
                weight = graph[u][v]
                # 如果 u -> v 有边 (weight != 0 且不是 inf)
                if weight != 0 and weight != np.inf:
                    # 如果通过 u 到达 v 的距离更短 , 则更新
                    if distances[u] + weight < distances[v]:
                        distances[v] = distances[u] + weight

    # 3 . 额外检测 : 第 n 轮 如果还能够更新 , 说明具有负权环
    for u in range(n):
        for v in range(n):
            weight = graph[u][v]
            if weight != 0 and weight != np.inf:
                if distances[u] + weight < distances[v]:
                    print("存在负权环")

    return distances                


INF = np.inf
# 带有负权边的图
my_graph = [
    [0,   1,   4,   INF],
    [INF, 0,  -2,   6],   # 1->2 是负权边 -2
    [INF, INF, 0,   3],
    [INF, INF, INF, 0]
]

dists = bellman_ford_custom(my_graph, 0)
print(f"从节点 0 出发的最短距离: {dists}")

从节点 0 出发的最短距离: [ 0.  1. -1.  2.]
